# Explore Mapped Hotel Data

This notebook reads the mapped Parquet data from MinIO and visualizes the hotel data.

**Data Source**: `s3a://data-lake/mapped_input/{country}/{supplier}/`

## 1. Setup Spark Session with MinIO Configuration

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, round as spark_round
import os

# Override any problematic environment variables
os.environ.pop("HADOOP_CONF_DIR", None)
os.environ.pop("YARN_CONF_DIR", None)

# Create Spark session with MinIO/S3A configuration
# Include necessary JAR packages for S3A support
spark = (
    SparkSession.builder.appName("Explore Mapped Hotel Data")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262",
    )
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider",
    )
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000")
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000")
    .config("spark.hadoop.fs.s3a.socket.send.buffer", "8192")
    .config("spark.hadoop.fs.s3a.socket.recv.buffer", "8192")
    .config("spark.hadoop.fs.s3a.attempts.maximum", "3")
    .config("spark.hadoop.fs.s3a.retry.limit", "3")
    .config("spark.hadoop.fs.s3a.retry.interval", "500")
    .config("spark.hadoop.fs.s3a.retry.throttle.limit", "3")
    .config("spark.hadoop.fs.s3a.retry.throttle.interval", "1000")
    .config("spark.hadoop.fs.s3a.connection.maximum", "50")
    .config("spark.hadoop.fs.s3a.threads.max", "10")
    .config("spark.hadoop.fs.s3a.threads.core", "5")
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60")
    .config("spark.hadoop.fs.s3a.max.total.tasks", "5")
    .config("spark.hadoop.fs.s3a.readahead.range", "65536")
    .config("spark.hadoop.fs.s3a.paging.maximum", "5")
    .config("spark.hadoop.fs.s3a.list.version", "2")
    .config("spark.hadoop.fs.s3a.committer.threads", "4")
    .config("spark.hadoop.fs.s3a.committer.staging.tmp.path", "/tmp/staging")
    .config("spark.hadoop.fs.s3a.buffer.dir", "/tmp")
    .config("spark.hadoop.fs.s3a.multipart.size", "104857600")
    .config("spark.hadoop.fs.s3a.multipart.threshold", "2147483647")
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400")
    .config("spark.hadoop.fs.s3a.fast.upload", "true")
    .config("spark.hadoop.fs.s3a.fast.upload.buffer", "disk")
    .config("spark.hadoop.fs.s3a.fast.upload.active.blocks", "4")
    .config("spark.hadoop.fs.s3a.block.size", "33554432")
    .config("spark.hadoop.fs.s3a.metadatastore.authoritative", "false")
    .config("spark.sql.files.maxPartitionBytes", "134217728")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("✓ Spark session created successfully")

:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/nakul.patil/.ivy2.5.2/cache
The jars for the packages stored in: /Users/nakul.patil/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-3a155423-a5aa-4605-b278-432649c82191;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
:: resolution report :: resolve 137ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final

✓ Spark session created successfully


## 2. Read Mapped Parquet Data

Specify the country and supplier to read data from:

In [ ]:
# Set parameters
country = "india"
supplier = "expedia"

# Define path
mapped_path = f"s3a://data-lake/mapped_input/{country}/{supplier}/"
print(f"Reading from: {mapped_path}")

# Read Parquet data
df = spark.read.parquet(mapped_path)

# Display basic info
print(f"\n✓ Data loaded successfully")
print(f"Total records: {df.count():,}")
print(f"Total columns: {len(df.columns)}")

Reading from: s3a://data-lake/mapped_input/india/expedia/


26/02/05 17:09:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties



✓ Data loaded successfully
Total records: 1,152
Total columns: 31


## 3. Schema Overview

In [3]:
# Display schema
print("Data Schema:")
df.printSchema()
df.show(5, truncate=False)

Data Schema:
root
 |-- hotel_id: string (nullable = true)
 |-- hotel_name: string (nullable = true)
 |-- relevance_score: string (nullable = true)
 |-- provider_id: string (nullable = true)
 |-- provider_hotel_id: string (nullable = true)
 |-- provider_name: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- address_line1: string (nullable = true)
 |-- address_line2: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- state_code: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- phone_numbers: string (nullable = true)
 |-- fax_numbers: string (nullable = true)
 |-- email_addresses: string (nullable = true)
 |-- hotel_type: string (nullable = true)
 |-- hotel_category: string (nullable = true)
 |-- star_rating: double (nullable = true)
 |-- distance: double (nullable =

26/02/05 17:09:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+---------------------------+---------------+-----------+-----------------+-------------+---------+---------+------------------------------------+---------------------+------+-----------+----------+------------+------------+-----------+---------------+---------------+---------------+----------+--------------+-----------+--------+-----------+-------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------